- `Acesso aos dados`: [Sentinel-5P NRTI CO: Near Real-Time Carbon Monoxide](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_NRTI_L3_CO)
- `Período dos dados`: 2018-11-22T12:00:13Z–2025-04-12T09:36:31Z
- `Resolução espacial`: 1113.2 metros
- `Resolução temporal`: diária
- `Código realizado por`: Enrique V. Mattos - 19/05/2024

# **1° Passo:** Preparando ambiente

In [ ]:
# instalando bibliotecas
!pip install -q ultraplot cartopy salem rasterio

# iniciando GEE e instalando XEE (transforma dados do GEE para formato DataSet)
!pip install -q eemont xee
import ee, geemap
ee.Authenticate()
ee.Initialize(project='ee-enrique', opt_url='https://earthengine-highvolume.googleapis.com')

# importa bibliotecas
import numpy as np
import ultraplot as uplt
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import time
from zipfile import ZipFile
import salem
from datetime import datetime
import glob
import xarray as xr
import os
import cartopy.crs as ccrs
import warnings
warnings.filterwarnings('ignore')

# monta drive
from google.colab import drive
drive.mount('/content/drive')

# caminho do drive
dir = '/content/drive/MyDrive/5_EXTENSAO/02_prefeitura_analise_periodo_chuvoso_2024_2025'

# leitura do shapefile do Brasil
shapefile_brasil = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/brasil/BRAZIL.shp')

# leitura do shapefile com a biblioteca SALEM
url = 'https://github.com/evmpython/shapefile/raw/main/'
shp = salem.read_shapefile(f'{url}itajuba/itajuba.shp')
itajuba = salem.read_shapefile(f'{url}itajuba/itajuba.shp')
mg = salem.read_shapefile(f'{url}estado_MG/MG_UF_2019.shp')

# limites do Brasil
lonmin_BR, lonmax_BR, latmin_BR, latmax_BR = -75.0, -33.0, -35.0, 7.0

# limites de MG
lonmin_MG, lonmax_MG, latmin_MG, latmax_MG = -52., -39., -23., -14.

# **PARTE 1):** Processamento

## 1) Mapa no GEE

In [ ]:
#========================================================================================================================#
#                                          FILTRA REGIÃO DE INTERESSE
#========================================================================================================================#
brasil = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
estado_mg = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Minas Gerais'))
municipio_itajuba = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

#========================================================================================================================#
#                                            CARREGA OS DADOS
#========================================================================================================================#
# carrega os dados. CO_column_number_density esta em unidades de "mol/m^2"
S5P_co = ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_CO') \
           .filter(ee.Filter.date('2025-02-01', '2025-03-01')) \
           .select('CO_column_number_density') \
           .filterBounds(estado_mg)

# constante de conversão mol/m² para DU
MOL_M2_TO_DU = 2240.4

#========================================================================================================================#
#                                           PLOTA FIGURA
#========================================================================================================================#
# cria a moldura do mapa
Map = geemap.Map()

# centraliza o mapa na região
Map.centerObject(estado_mg, zoom=6)

# parâmetros de visualização
vis = {'min': 0, 'max': 150, 'palette': ['black', 'blue', 'purple', 'cyan', 'green', 'yellow', 'red']}

# plota mapa
Map.addLayer(S5P_co.mean().clip(estado_mg).multiply(MOL_M2_TO_DU), vis, 'Média')
Map.addLayer(S5P_co.max().clip(estado_mg).multiply(MOL_M2_TO_DU), vis, 'Máximo')

# contorno da região
style1 = {'color': 'red', 'fillColor': '00000000'}
Map.addLayer(estado_mg.style(**style1), {}, 'MG')

# barra de cores
Map.add_colorbar_branca(colors=vis['palette'], vmin=vis['min'], vmax=vis['max'], layer_name='CO (DU)')

# exibe na tela
Map

In [ ]:
# mostra os dados
S5P_co

In [ ]:
# transforma a data para o formato "ano-mes-dia hora:minuto" e extrai os valores
S5P_co_2 = S5P_co.map(lambda img: img.set( {"DATE": ee.Date(img.get("system:time_start")).format("YYYY-MM-dd hh:mm")}))

# agrega as informações
agrega = (S5P_co_2.aggregate_array("DATE").getInfo())

# mostra na tela
pd.DataFrame(agrega)

## 2) Produz arquivo netcdf
- Demora `8 min` para processar 6 meses de dados

In [ ]:
%%time
# região de estudo
estado_mg = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Minas Gerais'))

# constante de conversão mol/m² para DU
MOL_M2_TO_DU = 2240.4

data_inicio = '2023-10-01'
data_fim = '2024-04-01'

# carrega coleção diária de CO (Sentinel-5P)
co_diario = ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_CO') \
              .filterDate(data_inicio, data_fim) \
              .select('CO_column_number_density') \
              .map(lambda img: img.multiply(MOL_M2_TO_DU).copyProperties(img, img.propertyNames()))

# função auxiliar para obter o primeiro dia de cada mês
def gerar_lista_meses(data_ini, data_fim):

    ini = ee.Date(data_ini)
    fim = ee.Date(data_fim)

    def gera_datas(n):
        return ini.advance(n, 'month')

    n_meses = fim.difference(ini, 'month').round()
    return ee.List.sequence(0, n_meses.subtract(1)).map(gera_datas)

# função para calcular a média mensal
def media_mensal(data):

    # data atual
    data = ee.Date(data)

    # data do próximo mês
    prox_mes = data.advance(1, 'month')

    # filtra o período considerando o mês atual
    colecao_mes = co_diario.filterDate(data, prox_mes)

    # calcula a média para o período
    media = colecao_mes.max().set({'system:time_start': data.millis(), 'mes': data.format('YYYY-MM')})

    return media

# criar ImageCollection com imagens mensais
datas_mensais = gerar_lista_meses(data_inicio, data_fim)
co_mensal = ee.ImageCollection(datas_mensais.map(media_mensal))

# converte para Dataset
ds_co_mensal = xr.open_dataset(co_mensal,
                               engine = 'ee',
                               crs = 'EPSG:4326',
                               scale = 0.10,
                               geometry = estado_mg.geometry())

# muda de "(time, lon, lat)" para "(time, lat, lon)"
ds_co_mensal = ds_co_mensal.transpose("time", "lat", "lon")

# salva para arquivo netcdf
ds_co_mensal.to_netcdf(f'{dir}/output/02_MONOXIDO_CARBONO/S5P_co_mensal_{data_inicio}_to_{data_fim}.nc')

In [ ]:
ds_co_mensal

In [ ]:
ds_co_mensal['CO_column_number_density'][0,:,:].plot(cmap='lajolla', vmin=50, vmax=150)

In [ ]:
ds_co_mensal['CO_column_number_density'][3,:,:].plot(cmap='lajolla', vmin=50, vmax=150)

# **PARTE 2):** Plota painel de figuras - `MAPA DE DENSIDADE`

In [ ]:
# meses
meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

# cria a moldura da figura
fig, ax = uplt.subplots(figsize=(18, 12.5), nrows=2, ncols=3, tight=True, proj='pcarree', sharex=True, sharey=True)

# formatação dos eixos
ax.format(coast=False, borders=False, innerborders=False,
          labels=False, latlines=5, lonlines=10,
          latlim=(latmin_MG, latmax_MG), lonlim=(lonmin_MG, lonmax_MG),
          suptitle=f'Monóxido de Carbono (DU)',
          small='20px', large='35px',
          linewidth=0, grid=False)

# leitura do arquivo netcdf
ds = xr.open_dataset(f'{dir}/output/02_MONOXIDO_CARBONO/S5P_co_mensal_{data_inicio}_to_{data_fim}.nc')

# loop dos meses
for i in range(0, ds.dims['time']):

    print('Processando ===>>>',  ds.time[i].values)

    # extraindo ano e mês com o accessor `.dt`
    ano = ds.time[i].dt.year.item()
    mes = ds.time[i].dt.month.item()

    print(f"Ano: {ano}, Mês: {mes}")

    # plota figura
    # opções de palltes: https://proplot.readthedocs.io/en/latest/colormaps.html
    map1 = ax[i].contourf(ds['lon'],
                          ds['lat'],
                          ds['CO_column_number_density'][i,:,:].salem.roi(shape=mg),
                          cmap='lajolla',
                          vmin=50, vmax=150,
                          levels=uplt.arange(50, 150, 10),
                          extend='max')

    # máximo de CO no estado
    #total = np.max(ds['CO_column_number_density'][i,:,:].salem.roi(shape=mg))
    #total = str(int(total.values))

    # plota titulo de cada figura
    #ax[i].format(title=f'{ano}-{mes}\n[CO={total} DU]', labels = False, titleloc='c', titlecolor='bright blue', fontsize=20)
    ax[i].format(title=f'{ano}-{str(mes).zfill(2)}', labels = False, titleloc='c', titlecolor='bright blue', fontsize=20)

    # plota contorno de MG e Itajubá
    mg.plot(edgecolor='black', facecolor='none', linewidth=1.2, alpha=1, ax=ax[i])
    itajuba.plot(edgecolor='red', facecolor='none', linewidth=0.5, alpha=1, ax=ax[i])

# informação na figura
ax[3].annotate('Prof. Enrique Mattos/UNIFEI\ngithub.com/evmpython', xy=(lonmin_MG, latmin_MG-4.0), fontsize=15, color='black')

# plota barra de cores da figura
fig.colorbar(map1, loc='b', label='Fonte: Sentinel 5P/Pixel: 10km', ticks=10, ticklabelsize=22, labelsize=22, space=0.5, length=0.60, width=0.4)

# salva figura
fig.savefig(f'{dir}/output/Fig_4_S5P_CO_MG.jpg', transparent=True, dpi=300, bbox_inches="tight", pad_inches=0.1)

In [ ]:
ds